In [57]:
import pandas as pd

tweets_trainset = pd.read_csv('data/TweetsTrainset.txt', sep='\t', header=None)
display(tweets_trainset.head(10))
print(f"Shape: {tweets_trainset.shape}")

,0,1,2
0,1,LOC/Thai Buddhist temple;,"2,000 fetuses found hidden at Thai Buddhist te..."
1,2,LOC/canada;,"870, 000 people in canada depend on #FakeHasht..."
2,3,PER/louis;,"7961212234, phone this girl! she is like louis..."
3,4,ORG/WikiLeaks;LOC/Southern Ocean;,@FakeUsername : WikiLeaks Set To Reveal US-UFO...
4,5,ORG/queen;,"@FakeUsername queen, bohemian rhapsody please"
5,6,PER/cheryl cole;PER/Danni;PER/eddie;,cheryl cole is starting to lose that connectio...
6,7,ORG/Lester;MISC/Mason & Begg Limited;,"Lester, '' Mason & Begg Limited : '' '' `Forbi..."
7,8,PER/Thomas Watson;,"To be successful, you have to have your heart ..."
8,9,"PER/Sanchez;PER/Eli Wallach;MISC/The Good, The...","Sanchez looks like Eli Wallach in The Good, Th..."
9,10,NaN,"@FakeUsername the history of the mayor, the ci..."


Shape: (2815, 3)


In [58]:
import re
from collections import Counter
import nltk
from nltk.tokenize import word_tokenize

try:
    nltk.data.find('tokenizers/punkt')
except LookupError:
    nltk.download('punkt')

# BIO Tagging Alignment
print("BIO Tagging Alignment\n" + "="*70)

# Helper function: Find entity positions in text
def find_entities(text, entities_dict):
    """Find character positions of all entities in text"""
    entity_spans = {}
    for ent_type, names in entities_dict.items():
        entity_spans[ent_type] = []
        for name in names:
            pattern = re.compile(re.escape(name), re.IGNORECASE)
            for match in pattern.finditer(text):
                entity_spans[ent_type].append((match.start(), match.end()))
    return entity_spans

# Helper function: Parse entity string
def parse_entities(ent_str):
    """Parse 'TYPE/entity; ' format"""
    entities = {}
    if pd.isna(ent_str) or not str(ent_str).strip():
        return entities
    
    for pair in str(ent_str).split(';'):
        if '/' in pair:
            ent_type, name = pair.split('/', 1)
            ent_type = ent_type.strip()
            name = name.strip()
            if ent_type not in entities:
                entities[ent_type] = []
            entities[ent_type].append(name)
    return entities

# Helper function: Assign BIO tags
def get_bio_tags(tokens, positions, entity_spans):
    """Assign BIO tags to tokens based on entity positions"""
    tags = []
    for start, end, token in positions:
        tag = 'O'
        
        for ent_type, spans in entity_spans.items():
            for ent_start, ent_end in spans:
                if start >= ent_start and end <= ent_end:
                    tag = 'B-' + ent_type if start == ent_start else 'I-' + ent_type
                    break
            if tag != 'O':
                break
        tags.append(tag)
    return tags

# Process all sequences
sequences = []
for idx, row in tweets_trainset.iterrows():
    text = row[2]
    
    # Get NLTK tokens with positions
    nltk_tokens = word_tokenize(text)
    tokens_pos = []
    pos = 0
    for token in nltk_tokens:
        start = text.find(token, pos)
        if start == -1:
            start = pos
        end = start + len(token)
        tokens_pos.append((start, end, token))
        pos = end
    
    # Parse and find entities
    entities = parse_entities(row[1])
    entity_spans = find_entities(text, entities)
    
    # Get tokens and tags
    tokens = [t for _, _, t in tokens_pos]
    tags = get_bio_tags(tokens, tokens_pos, entity_spans)
    
    if tokens:
        sequences.append({
            'text': text,
            'tokens': tokens,
            'tags': tags,
            'num_tokens': len(tokens)
        })


sequences_df = pd.DataFrame(sequences)

print(f"\nTotal sequences: {len(sequences_df):,}")

# Sample sequences
print("\n" + "-"*70)
print("SAMPLE SEQUENCES:\n")
for i in range(min(3, len(sequences_df))):
    s = sequences_df.iloc[i]
    print(f"Sequence {i+1}:")
    print(f"  Text:   {s['text']}")
    print(f"  Tokens: {s['tokens']}")
    print(f"  Tags:   {s['tags']}")
    print()

display(sequences_df.head(10))


BIO Tagging Alignment

Total sequences: 2,815

----------------------------------------------------------------------
SAMPLE SEQUENCES:

Sequence 1:
  Text:   2,000 fetuses found hidden at Thai Buddhist temple http://FakeURL via @FakeUsername
  Tokens: ['2,000', 'fetuses', 'found', 'hidden', 'at', 'Thai', 'Buddhist', 'temple', 'http', ':', '//FakeURL', 'via', '@', 'FakeUsername']
  Tags:   ['O', 'O', 'O', 'O', 'O', 'B-LOC', 'I-LOC', 'I-LOC', 'O', 'O', 'O', 'O', 'O', 'O']

Sequence 2:
  Text:   870, 000 people in canada depend on #FakeHashtag -25% increase in the last 2 years - please give generously
  Tokens: ['870', ',', '000', 'people', 'in', 'canada', 'depend', 'on', '#', 'FakeHashtag', '-25', '%', 'increase', 'in', 'the', 'last', '2', 'years', '-', 'please', 'give', 'generously']
  Tags:   ['O', 'O', 'O', 'O', 'O', 'B-LOC', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O']

Sequence 3:
  Text:   7961212234, phone this girl! she is like louis biggest fa

,text,tokens,tags,num_tokens
0,"2,000 fetuses found hidden at Thai Buddhist te...","[2,000, fetuses, found, hidden, at, Thai, Budd...","[O, O, O, O, O, B-LOC, I-LOC, I-LOC, O, O, O, ...",14
1,"870, 000 people in canada depend on #FakeHasht...","[870, ,, 000, people, in, canada, depend, on, ...","[O, O, O, O, O, B-LOC, O, O, O, O, O, O, O, O,...",22
2,"7961212234, phone this girl! she is like louis...","[7961212234, ,, phone, this, girl, !, she, is,...","[O, O, O, O, O, O, O, O, O, B-PER, O, O, O, O,...",23
3,@FakeUsername : WikiLeaks Set To Reveal US-UFO...,"[@, FakeUsername, :, WikiLeaks, Set, To, Revea...","[O, O, O, B-ORG, O, O, O, O, O, O, B-LOC, I-LO...",29
4,"@FakeUsername queen, bohemian rhapsody please","[@, FakeUsername, queen, ,, bohemian, rhapsody...","[O, O, B-ORG, O, O, O, O]",7
5,cheryl cole is starting to lose that connectio...,"[cheryl, cole, is, starting, to, lose, that, c...","[B-PER, I-PER, O, O, O, O, O, O, O, O, O, O, B...",28
6,"Lester, '' Mason & Begg Limited : '' '' `Forbi...","[Lester, ,, ``, Mason, &, Begg, Limited, :, ``...","[B-ORG, O, O, B-MISC, I-MISC, I-MISC, I-MISC, ...",33
7,"To be successful, you have to have your heart ...","[To, be, successful, ,, you, have, to, have, y...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ...",26
8,"Sanchez looks like Eli Wallach in The Good, Th...","[Sanchez, looks, like, Eli, Wallach, in, The, ...","[B-PER, O, O, B-PER, I-PER, O, B-MISC, I-MISC,...",19
9,"@FakeUsername the history of the mayor, the ci...","[@, FakeUsername, the, history, of, the, mayor...","[O, O, O, O, O, O, O, O, O, O, O]",11


In [59]:
from sklearn.model_selection import train_test_split

# Train/Validation Split
print("Data Split - Train/Validation\n" + "="*70)

# Split: 80% train, 20% validation
train_df, val_df = train_test_split(sequences_df, test_size=0.2, random_state=42, shuffle=True)

def df_to_ner_format(df):
    """Convert DataFrame to list of (tokens, tags) for NER training"""
    return [(row['tokens'], row['tags']) for _, row in df.iterrows()]

train_data = df_to_ner_format(train_df)
val_data = df_to_ner_format(val_df)

print(f"Total sequences: {len(sequences_df):,}")
print(f"Training set (80%): {len(train_df):,} sequences")
print(f"Validation set (20%): {len(val_df):,} sequences")

train_true_tags = [row['tags'] for _, row in train_df.iterrows()]
val_true_tags = [row['tags'] for _, row in val_df.iterrows()]

print(f"\nTraining tags: {len(train_true_tags):,} sequences")
print(f"Validation tags: {len(val_true_tags):,} sequences")

#Check
display(train_df.head(10))
sample_tokens, sample_tags = train_data[0]
print("\nSample from Training Set:")
print(f"Tokens: {sample_tokens}")
print(f"Tags:   {sample_tags}")

Data Split - Train/Validation
Total sequences: 2,815
Training set (80%): 2,252 sequences
Validation set (20%): 563 sequences

Training tags: 2,252 sequences
Validation tags: 563 sequences


,text,tokens,tags,num_tokens
1744,Hanie gon get all that Chi Town ass he pull th...,"[Hanie, gon, get, all, that, Chi, Town, ass, h...","[B-PER, O, O, O, O, B-LOC, I-LOC, O, O, O, O, ...",13
2186,Larry Moore recording a late promo ahead of @F...,"[Larry, Moore, recording, a, late, promo, ahea...","[B-PER, I-PER, O, O, O, O, O, O, O, O, O, O, O...",17
2371,Natalie Portman : Her Evolution From Eerily Ma...,"[Natalie, Portman, :, Her, Evolution, From, Ee...","[B-PER, I-PER, O, O, O, O, O, O, O, O, O, O, O...",24
2514,People are smarter than you think . Give them ...,"[People, are, smarter, than, you, think, ., Gi...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ...",18
1025,@FakeUsername The fun - and the money - never ...,"[@, FakeUsername, The, fun, -, and, the, money...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ...",17
2401,Nimrod Laments http://FakeURL,"[Nimrod, Laments, http, :, //FakeURL]","[B-PER, I-PER, O, O, O]",5
644,"@FakeUsername Ima nerd, nerds aint tough right?","[@, FakeUsername, Ima, nerd, ,, nerds, aint, t...","[O, O, B-PER, O, O, O, O, O, O, O]",10
2252,Lol . I thought u missed that RT @FakeUsername...,"[Lol, ., I, thought, u, missed, that, RT, @, F...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ...",36
461,@FakeUsername #FakeHashtag #FakeHashtag GoI ...,"[@, FakeUsername, #, FakeHashtag, #, FakeHasht...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ...",31
1868,I commented on a YouTube video http://FakeURL,"[I, commented, on, a, YouTube, video, http, :,...","[O, O, O, O, B-ORG, O, O, O, O]",9



Sample from Training Set:
Tokens: ['Hanie', 'gon', 'get', 'all', 'that', 'Chi', 'Town', 'ass', 'he', 'pull', 'this', 'one', 'off']
Tags:   ['B-PER', 'O', 'O', 'O', 'O', 'B-LOC', 'I-LOC', 'O', 'O', 'O', 'O', 'O', 'O']


In [60]:
try:
    from seqeval.metrics import precision_score, recall_score, f1_score, classification_report
except ImportError:
    print("seqeval not found.")
    %pip install seqeval

from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix
try:
    import matplotlib.pyplot as plt
except ImportError:
    print("matplotlib not found.")
    %pip install matplotlib

# Evaluation function
def evaluate_ner(y_true, y_pred, set_name="Validation"):
    """
    Evaluate NER using seqeval (works directly with BIO tags!)
    
    Args:
        y_true: List of tag sequences [['B-PER', 'I-PER', 'O']]
        y_pred: List of predicted tag sequences
    """
    print(f"\nEvaluation Results for {set_name} Set\n" + "-"*70)

    # Calculate metrics
    precision = precision_score(y_true, y_pred)
    recall = recall_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)

    print(f"Precision (Micro): {precision:.4f}")
    print(f"Recall (Micro):    {recall:.4f}")
    print(f"F1-Score (Micro):  {f1:.4f}")
    print("\nClassification Report:")
    print(classification_report(y_true, y_pred))

    return {'precision': precision, 'recall': recall, 'f1': f1}

def plot_confusion_matrix(y_true, y_pred, set_name="Validation"):
    """Plot confusion matrix for BIO tags"""
    # Flatten lists
    y_true_flat = [tag for seq in y_true for tag in seq]
    y_pred_flat = [tag for seq in y_pred for tag in seq]

    all_tags = sorted(set(y_true_flat + y_pred_flat))
    cm = confusion_matrix(y_true_flat, y_pred_flat, labels=all_tags)

    fig, ax = plt.subplots(figsize=(12, 8))
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels = all_tags)
    disp.plot(ax=ax, cmap='Blues', values_format='d')
    plt.title(f"Confusion Matrix - {set_name} Set")
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

print("Evaluation Framework Finished\n")

Evaluation Framework Finished



In [61]:
print("METHOD A - Rule-Based + CRF Approach\n" + "="*70)

import nltk
from nltk import pos_tag
from nltk.tokenize import word_tokenize

try:
    nltk.data.find('taggers/averaged_perceptron_tagger')
except LookupError:
    nltk.download('averaged_perceptron_tagger')

print ("Feature Extraction - VARIATION A1: Basic Features\n" + "-"*70)

def word2features_A1(word, pos):
    """ Extract 11 basic features for a word.
    
    word: The word token
    pos: Part-of-speech tag of the word

    Returns a dictionary of features for sklearn-crfsuite.
    """

    features_A1 = {
        'bias' : 1.0,  # Bias term to help the model learn a baseline
        'word.lower()': word.lower(), 
        'len(word)': len(word),  # Length of the word
        'word[-2:]': word[-2:],  # Last 2 characters (suffix)
        'word[:2]': word[:2],  # First 2 characters (prefix)
        'word.isupper()': word.isupper(),  # Is the word in uppercase?
        'word.islower()': word.islower(),  # Is the word in lowercase?
        'word.istitle()': word.istitle(),  # Is the word title
        'word.isdigit()': word.isdigit(),  # Is the word a digit?
        'postag': pos,  # Tag
        'is_mention': word.startswith('@'),  # Is it a Twitter mention?
        'is_hashtag': word.startswith('#'),  # Is it a Twitter hashtag?
    }

    return features_A1

def sent2features_A1(sent):
    """
    Extract features for all words in a sentence.
    POS tag the entire sentence at once for efficiency.
    
    sent: List of tokens
    
    Returns: List of feature dictionaries (one per word)
    """
    # POS tag entire sentence at once
    pos_tags = pos_tag(sent) # Returns [(word, tag), (word, tag)]
    # Extract features for each word with its POS tag
    return [word2features_A1(word, pos) for word, pos in pos_tags]

print("\nExtracting features from TRAINING set...")
X_train_A1 = [sent2features_A1(tokens) for tokens, tags in train_data]
y_train_A1 = [tags for tokens, tags in train_data]

print(f"X_train_A1: {len(X_train_A1):,} sequences")
print(f"y_train_A1: {len(y_train_A1):,} sequences")

print("\nExtracting features from VALIDATION set...")
X_val_A1 = [sent2features_A1(tokens) for tokens, tags in val_data]
y_val_A1 = [tags for tokens, tags in val_data]

print(f"X_val_A1: {len(X_val_A1):,} sequences")
print(f"y_val_A1: {len(y_val_A1):,} sequences")

# Sample feature output
print("\n" + "="*70)
print("\nSample features for training sequence:")
sample_seq_idx_A1 = 20
sample_tokens_A1, sample_tags_A1 = train_data[sample_seq_idx_A1]
sample_features_A1 = X_train_A1[sample_seq_idx_A1]

print(f"Text: {' '.join(sample_tokens_A1)}")
print(f"Tags: {sample_tags_A1}\n")

for word_idx in range(len(sample_tokens_A1)):
    print(f"Word {word_idx}: '{sample_tokens_A1[word_idx]}' -> Tag: {sample_tags_A1[word_idx]}")
    print(f"  Features: {sample_features_A1[word_idx]}")
    print()

METHOD A - Rule-Based + CRF Approach
Feature Extraction - VARIATION A1: Basic Features
----------------------------------------------------------------------

Extracting features from TRAINING set...
X_train_A1: 2,252 sequences
y_train_A1: 2,252 sequences

Extracting features from VALIDATION set...
X_val_A1: 563 sequences
y_val_A1: 563 sequences


Sample features for training sequence:
Text: @ FakeUsername yes , here we get blizzards , hurricanes and an occasional visit by Ann Murray .
Tags: ['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'B-PER', 'I-PER', 'O']

Word 0: '@' -> Tag: O
  Features: {'bias': 1.0, 'word.lower()': '@', 'len(word)': 1, 'word[-2:]': '@', 'word[:2]': '@', 'word.isupper()': False, 'word.islower()': False, 'word.istitle()': False, 'word.isdigit()': False, 'postag': 'JJ', 'is_mention': True, 'is_hashtag': False}

Word 1: 'FakeUsername' -> Tag: O
  Features: {'bias': 1.0, 'word.lower()': 'fakeusername', 'len(word)': 12, 'word[-2:]': 'me'

In [62]:
print ("Feature Extraction - VARIATION A2: Contextual Features\n" + "-"*70)

def word2features_A2(sent, i, pos_tags):
    """
    Extract contextual features for word at position i.
    Includes: current word features + previous and next word features + BOS/EOS.

    sent: List of tokens in the sentence
    i: Index of target word
    pos_tags: List of (word, POS) tuples from pos_tag(sent)

    Return: Dictionary of features for the word at position i
    """

    word = sent[i]
    pos = pos_tags[i][1]  # POS tag for current word
    
    features_A2 = {
        'bias': 1.0,
        'word.lower()': word.lower(),
        'len(word)': len(word),  # Length of the word
        'word[-2:]': word[-2:],  # Last 2 characters (suffix)
        'word[:2]': word[:2],  # First 2 characters (prefix)
        'word.isupper()': word.isupper(),
        'word.islower()': word.islower(),
        'word.istitle()': word.istitle(),
        'word.isdigit()': word.isdigit(),
        'postag': pos,
        'is_mention': word.startswith('@'),
        'is_hashtag': word.startswith('#'),
    }

    # Previous word features
    if i > 0:
        prev_word = sent[i-1]
        prev_pos = pos_tags[i-1][1]
        features_A2.update({
            'prev_word.lower()': prev_word.lower(),
            'prev_word.isupper()': prev_word.isupper(),
            'prev_word.isdigit()': prev_word.isdigit(),
            'prev_postag': prev_pos,
        })
    else:
        features_A2['BOS'] = True  # Beginning of sentence
        features_A2['prev_word.lower()'] = 'BOS'
        features_A2['prev_postag'] = 'BOS'
    
    # Next word features
    if i < len(sent) - 1:
        next_word = sent[i+1]
        next_pos = pos_tags[i+1][1]
        features_A2.update({
            'next_word.lower()': next_word.lower(),
            'next_word.isupper()': next_word.isupper(),
            'next_word.isdigit()': next_word.isdigit(),
            'next_postag': next_pos,
        })
    else:
        features_A2['EOS'] = True  # End of sentence
        features_A2['next_word.lower()'] = 'EOS'
        features_A2['next_postag'] = 'EOS'

    return features_A2

def sent2features_A2(sent):
    """
    Extract features for all words in a sentence using contextual features.
    
    sent: List of tokens
    
    Returns: List of feature dictionaries (one per word)
    """
    pos_tags = pos_tag(sent)
    return [word2features_A2(sent, i, pos_tags) for i in range(len(sent))]

print("\n Extracting contextual features from TRAINING set...")
X_train_A2 = [sent2features_A2(tokens) for tokens, tags in train_data]
y_train_A2 = [tags for tokens, tags in train_data]

print(f"X_train_A2: {len(X_train_A2):,} sequences")
print(f"y_train_A2: {len(y_train_A2):,} sequences")

print("\n Extracting contextual features from VALIDATION set...")
X_val_A2 = [sent2features_A2(tokens) for tokens, tags in val_data]
y_val_A2 = [tags for tokens, tags in val_data]

print(f"X_val_A2: {len(X_val_A2):,} sequences")
print(f"y_val_A2: {len(y_val_A2):,} sequences")

# Sample feature output
print ("\n" + "="*70)
print("\nSample contextual features for training sequence:")
sample_seq_idx_A2 = 20
sample_tokens_A2, sample_tags_A2 = train_data[sample_seq_idx_A2]
sample_features_A2 = X_train_A2[sample_seq_idx_A2]

print(f"Text: {' '.join(sample_tokens_A2)}")
print(f"Tags: {sample_tags_A2}\n")

for word_idx in range(len(sample_tokens_A2)):
    print(f"Word {word_idx}: '{sample_tokens_A2[word_idx]}' -> Tag: {sample_tags_A2[word_idx]}")
    print(f"  Features: {sample_features_A2[word_idx]}")
    print()

Feature Extraction - VARIATION A2: Contextual Features
----------------------------------------------------------------------

 Extracting contextual features from TRAINING set...
X_train_A2: 2,252 sequences
y_train_A2: 2,252 sequences

 Extracting contextual features from VALIDATION set...
X_val_A2: 563 sequences
y_val_A2: 563 sequences


Sample contextual features for training sequence:
Text: @ FakeUsername yes , here we get blizzards , hurricanes and an occasional visit by Ann Murray .
Tags: ['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'B-PER', 'I-PER', 'O']

Word 0: '@' -> Tag: O
  Features: {'bias': 1.0, 'word.lower()': '@', 'len(word)': 1, 'word[-2:]': '@', 'word[:2]': '@', 'word.isupper()': False, 'word.islower()': False, 'word.istitle()': False, 'word.isdigit()': False, 'postag': 'JJ', 'is_mention': True, 'is_hashtag': False, 'BOS': True, 'prev_word.lower()': 'BOS', 'prev_postag': 'BOS', 'next_word.lower()': 'fakeusername', 'next_word.isupper()': 

In [63]:
# Compare feature set sizes
print("\nFeature Comparison (A1 vs A2):\n")

sample_seq_idx = 20
sample_tokens, sample_tags = train_data[sample_seq_idx]
sample_features_A1 = X_train_A1[sample_seq_idx]
sample_features_A2 = X_train_A2[sample_seq_idx]

print(f"Text: {' '.join(sample_tokens)}")
print(f"Tags: {sample_tags}\n")

# Show first word
word_idx = 1
print(f"Word {word_idx}: '{sample_tokens[word_idx]}' -> Tag: {sample_tags[word_idx]}")
print(f"\nA1 Features ({len(sample_features_A1[word_idx])} features):")
for key, val in sample_features_A1[word_idx].items():
    print(f"  {key}: {val}")

print(f"\nA2 Features ({len(sample_features_A2[word_idx])} features):")
for key, val in sample_features_A2[word_idx].items():
    print(f"  {key}: {val}")


Feature Comparison (A1 vs A2):

Text: @ FakeUsername yes , here we get blizzards , hurricanes and an occasional visit by Ann Murray .
Tags: ['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'B-PER', 'I-PER', 'O']

Word 1: 'FakeUsername' -> Tag: O

A1 Features (12 features):
  bias: 1.0
  word.lower(): fakeusername
  len(word): 12
  word[-2:]: me
  word[:2]: Fa
  word.isupper(): False
  word.islower(): False
  word.istitle(): False
  word.isdigit(): False
  postag: NNP
  is_mention: False
  is_hashtag: False

A2 Features (20 features):
  bias: 1.0
  word.lower(): fakeusername
  len(word): 12
  word[-2:]: me
  word[:2]: Fa
  word.isupper(): False
  word.islower(): False
  word.istitle(): False
  word.isdigit(): False
  postag: NNP
  is_mention: False
  is_hashtag: False
  prev_word.lower(): @
  prev_word.isupper(): False
  prev_word.isdigit(): False
  prev_postag: JJ
  next_word.lower(): yes
  next_word.isupper(): False
  next_word.isdigit(): False
  next_posta

In [64]:
%pip install sklearn-crfsuite

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [sklearn-crfsuite]
Note: you may need to restart the kernel to use updated packages.
